Шаг 1: Установка Spark

In [ ]:
# === Установка Spark в Google Colab (работает в 2026) ===
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.3/spark-3.5.3-bin-hadoop3.tgz
!tar xf spark-3.5.3-bin-hadoop3.tgz
!pip install -q findspark pyspark==3.5.3

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.3-bin-hadoop3"

import findspark
findspark.init()

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("EndomondoML_Variant9") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

print("Spark успешно запущен!")
spark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 11.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.2 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.3 which is incompatible.
Spark успешно запущен!


Шаг 2: Скачивание данных

In [ ]:
# Скачиваем данные (рекомендуемый способ)
import urllib.request
url = "https://mcauleylab.ucsd.edu/public_datasets/gdrive/fitrec/endomondoHR_proper.json"
filename = "endomondoHR.json"

print("Скачивание началось... Это может занять 5–15 минут.")
urllib.request.urlretrieve(url, filename)
print("Скачивание завершено!")

Скачивание началось... Это может занять 5–15 минут.
Скачивание завершено!


Шаг 3: Загрузка и базовая предобработка
Python

In [ ]:
from pyspark.sql.functions import col, size, when, hour, dayofweek, unix_timestamp, to_timestamp

df = spark.read.json("endomondoHR.json")

print(f"Исходное количество записей: {df.count():,}")
df.printSchema()

# Очистка
df_clean = df.filter(size(col("heart_rate")) > 10) \
             .filter(col("sport").isNotNull()) \
             .filter(col("timestamp").isNotNull()) \
             .dropna(subset=["heart_rate", "speed", "altitude"])

print(f"После очистки: {df_clean.count():,} записей")

Исходное количество записей: 167,783
root
 |-- altitude: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- gender: string (nullable = true)
 |-- heart_rate: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- id: long (nullable = true)
 |-- latitude: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- longitude: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- speed: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- sport: string (nullable = true)
 |-- timestamp: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- url: string (nullable = true)
 |-- userId: long (nullable = true)

После очистки: 31,673 записей


Шаг 4: Feature Engineering

In [ ]:
# === 4. Feature Engineering (ФИНАЛЬНАЯ РАБОЧАЯ ВЕРСИЯ) ===

from pyspark.sql.functions import col, size, when, hour, dayofweek, from_unixtime, expr

print("🔧 Создание признаков...")

# 1. Длительность тренировки в минутах
df_fe = df_clean.withColumn("duration_minutes",
                            (size(col("timestamp")) / 60.0).cast("double"))

# 2. Время начала тренировки
df_fe = df_fe.withColumn("start_ts", col("timestamp").getItem(0).cast("long"))
df_fe = df_fe.withColumn("hour_start", hour(from_unixtime(col("start_ts"))))

# 3. День недели (как double)
df_fe = df_fe.withColumn("day_of_week", dayofweek(from_unixtime(col("start_ts"))).cast("double"))

# 4. Время суток: утро / день / вечер
df_fe = df_fe.withColumn("time_of_day",
    when((col("hour_start") >= 5) & (col("hour_start") <= 11), "morning")
    .when((col("hour_start") >= 12) & (col("hour_start") <= 17), "day")
    .otherwise("evening")
)

# 5. Средняя скорость — исправленный aggregate (работает в Spark 3.5)
df_fe = df_fe.withColumn("avg_speed",
    expr("""
        aggregate(
            transform(speed, x -> cast(x as double)),
            cast(0.0 as double),
            (acc, x) -> acc + x,
            acc -> acc / size(speed)
        )
    """)
)

# Финальный датасет
df_final = df_fe.select(
    col("sport").alias("label"),
    col("duration_minutes"),
    col("time_of_day"),
    col("day_of_week"),
    col("avg_speed")
).na.drop()

print("✅ Feature Engineering успешно завершён!")
print(f"Количество записей после обработки: {df_final.count():,}")

# Проверка результата
df_final.show(5, truncate=False)
print("\nРаспределение по видам спорта:")
df_final.groupBy("label").count().orderBy("count", ascending=False).show(10)

🔧 Создание признаков...
✅ Feature Engineering успешно завершён!
Количество записей после обработки: 31,673
+----------------+-----------------+-----------+-----------+------------------+
|label           |duration_minutes |time_of_day|day_of_week|avg_speed         |
+----------------+-----------------+-----------+-----------+------------------+
|bike            |8.333333333333334|day        |1.0        |26.162157600000004|
|bike            |8.333333333333334|evening    |7.0        |27.218368800000004|
|bike            |8.333333333333334|day        |3.0        |26.050773600000017|
|bike            |8.333333333333334|day        |5.0        |26.87783759999997 |
|bike (transport)|8.333333333333334|day        |3.0        |29.592280800000022|
+----------------+-----------------+-----------+-----------+------------------+
only showing top 5 rows


Распределение по видам спорта:
+--------------------+-----+
|               label|count|
+--------------------+-----+
|                bike|17093|


Шаг 5: Построение ML Pipeline + Random Forest

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Индексация целевой переменной (sport)
label_indexer = StringIndexer(inputCol="label", outputCol="label_index")

# OneHot для категориальных признаков
time_indexer = StringIndexer(inputCol="time_of_day", outputCol="time_index")
time_encoder = OneHotEncoder(inputCol="time_index", outputCol="time_vec")

# VectorAssembler
assembler = VectorAssembler(
    inputCols=["duration_minutes", "day_of_week", "avg_speed", "time_vec"],
    outputCol="features_raw"
)

scaler = StandardScaler(inputCol="features_raw", outputCol="features")

# Модель Random Forest
rf = RandomForestClassifier(
    labelCol="label_index",
    featuresCol="features",
    numTrees=50,
    maxDepth=10,
    seed=42
)

# Полный Pipeline
pipeline = Pipeline(stages=[
    label_indexer,
    time_indexer,
    time_encoder,
    assembler,
    scaler,
    rf
])

# Разделение на train/test
train_data, test_data = df_final.randomSplit([0.8, 0.2], seed=42)

# Обучение модели
print("Обучение модели... (может занять 2–5 минут)")
model = pipeline.fit(train_data)

# Предсказания
predictions = model.transform(test_data)

Обучение модели... (может занять 2–5 минут)


Шаг 6: Оценка качества модели

In [ ]:
# Метрики
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label_index",
    predictionCol="prediction",
    metricName="accuracy"
)

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label_index",
    predictionCol="prediction",
    metricName="f1"
)

accuracy = evaluator_acc.evaluate(predictions)
f1_score = evaluator_f1.evaluate(predictions)

print(f"Accuracy:  {accuracy:.4f}")
print(f"F1-Score:  {f1_score:.4f}")

# Матрица ошибок (примерно)
predictions.groupBy("label", "prediction").count().show()

Accuracy:  0.8824
F1-Score:  0.8376
+--------------------+----------+-----+
|               label|prediction|count|
+--------------------+----------+-----+
|                 run|       1.0| 2246|
|      indoor cycling|       0.0|  151|
|                 run|       0.0|   65|
|       mountain bike|       0.0|  157|
|        orienteering|       1.0|    4|
|            kayaking|       1.0|    2|
|                bike|       0.0| 3312|
|       mountain bike|       1.0|   28|
|    bike (transport)|       1.0|    3|
|    bike (transport)|       0.0|  226|
|core stability tr...|       0.0|    4|
|                bike|       1.0|   59|
|                walk|       1.0|   17|
|      indoor cycling|       1.0|    4|
|cross-country skiing|       1.0|    2|
|core stability tr...|       1.0|    3|
|     fitness walking|       1.0|    5|
|       roller skiing|       0.0|    3|
|       roller skiing|       1.0|    3|
|              hiking|       1.0|    2|
+--------------------+----------+-----+
only

Шаг 7: Feature Importance

In [ ]:
rf_model = model.stages[-1]  # последний этап — RandomForest

importances = rf_model.featureImportances
feature_names = ["duration_minutes", "day_of_week", "avg_speed", "time_vec_0", "time_vec_1", "time_vec_2"]

print("Важность признаков:")
for name, imp in zip(feature_names, importances):
    print(f"{name:20} : {imp:.4f}")

Важность признаков:
duration_minutes     : 0.0000
day_of_week          : 0.0056
avg_speed            : 0.9840
time_vec_0           : 0.0070
time_vec_1           : 0.0034
